# Use case 5 — Polymorphic associations: enumeration and reverse range lookup

**Questions:**

1. *Enumeration.* Which USDM v4 attributes have a polymorphic range —
   `owl:unionOf` over multiple classes — and which classes do they target?
2. *Reverse range.* Given a target class, which attributes can hold it,
   directly or via polymorphism?
3. *Shadow visibility.* Are there classes reachable from object properties
   only through polymorphism — invisible to a consumer that walks direct
   ranges only?

**Pinned to v0.4.0.** Reads object-property ranges from `usdm_v4.ttl` and
traverses `owl:unionOf` RDF Collections to enumerate member classes.

In [1]:
import rdflib
import pandas as pd
from pathlib import Path

TTL_PATH = "../usdm_v4.ttl"
EXPECTED_BASELINE = 8642

if not Path(TTL_PATH).exists():
    raise FileNotFoundError(
        f"{TTL_PATH} missing — run notebooks/10/20 first to regenerate."
    )

g = rdflib.Graph()
g.parse(TTL_PATH, format="turtle")
n_triples = len(g)
print(f"parsed {n_triples} triples from {TTL_PATH}")
if n_triples != EXPECTED_BASELINE:
    print(f"  note: triple count differs from v0.4.0 baseline ({EXPECTED_BASELINE})"
          " — likely a DDF-RA tag bump; this notebook is pinned to v0.4.0 anchor shape")

def short(uri):
    """Last path segment, fragment-aware."""
    if uri is None:
        return None
    s = str(uri)
    if "#" in s:
        return s.split("#")[-1]
    return s.split("/")[-1]

parsed 8642 triples from ../usdm_v4.ttl


## Step 1 — enumerate polymorphic associations

A polymorphic attribute has `rdfs:range` pointing at a blank node whose
`owl:unionOf` lists multiple class members. SPARQL traverses the RDF
Collection (`rdf:first` / `rdf:rest`) to surface the members.

In [2]:
ENUMERATE_SPARQL = """
PREFIX owl:  <http://www.w3.org/2002/07/owl#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?prop ?declaring ?member
WHERE {
  ?prop a owl:ObjectProperty ;
        rdfs:domain ?declaring ;
        rdfs:range ?range .
  FILTER(isBlank(?range))
  ?range owl:unionOf ?list .
  ?list rdf:rest*/rdf:first ?member .
}
"""

rows = []
for r in g.query(ENUMERATE_SPARQL):
    rows.append({
        "attribute":       short(r["prop"]),
        "declaring_class": short(r["declaring"]),
        "member_class":    short(r["member"]),
    })

members_long = pd.DataFrame(rows).sort_values(["attribute", "member_class"]).reset_index(drop=True)

polymorphic_df = (
    members_long
    .groupby(["attribute", "declaring_class"], as_index=False)
    .agg(member_classes=("member_class", lambda s: sorted(s.tolist())),
         union_size=("member_class", "count"))
    .sort_values("union_size", ascending=False)
    .reset_index(drop=True)
)

print(f"{len(polymorphic_df)} polymorphic associations")
polymorphic_df

4 polymorphic associations


,attribute,declaring_class,member_classes,union_size
0,Condition-appliesToIds,Condition,"[Activity, BiomedicalConcept, BiomedicalConcep...",5
1,Condition-contextIds,Condition,"[Activity, ScheduledActivityInstance]",2
2,ProductOrganizationRole-appliesToIds,ProductOrganizationRole,"[AdministrableProduct, MedicalDevice]",2
3,StudyRole-appliesToIds,StudyRole,"[StudyDesign, StudyVersion]",2


## Step 2 — reverse range lookup

For a target class, find every object property whose range either points
at the class directly or includes it inside an `owl:unionOf`. The query
tags each row with `direct` or `via_union`.

Two demo classes — both broadly relevant in USDM:

- `Activity` — central to the SoA; expected 7 attributes (5 direct + 2 via union).
- `BiomedicalConcept` — central to measurement specification; expected 4
  attributes (3 direct + 1 via union).

In [3]:
REVERSE_SPARQL_TEMPLATE = """
PREFIX owl:  <http://www.w3.org/2002/07/owl#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX usdm: <https://w3id.org/cdisc/usdm/v4/>

SELECT ?prop ?declaring ?range_kind
WHERE {{
  ?prop rdfs:domain ?declaring .
  {{
    ?prop a owl:ObjectProperty ; rdfs:range usdm:{cls} .
    BIND("direct" AS ?range_kind)
  }} UNION {{
    ?prop a owl:ObjectProperty ; rdfs:range ?r .
    FILTER(isBlank(?r))
    ?r owl:unionOf ?list .
    ?list rdf:rest*/rdf:first usdm:{cls} .
    BIND("via_union" AS ?range_kind)
  }}
}}
"""

def attributes_targeting(target_class):
    q = REVERSE_SPARQL_TEMPLATE.format(cls=target_class)
    rows = [{
        "attribute":       short(r["prop"]),
        "declaring_class": short(r["declaring"]),
        "range_kind":      str(r["range_kind"]),
    } for r in g.query(q)]
    return (pd.DataFrame(rows)
            .sort_values(["range_kind", "attribute"])
            .reset_index(drop=True))

for demo_cls in ("Activity", "BiomedicalConcept"):
    df = attributes_targeting(demo_cls)
    n_direct = (df.range_kind == "direct").sum()
    n_union  = (df.range_kind == "via_union").sum()
    print(f"{demo_cls}: {len(df)} attributes ({n_direct} direct + {n_union} via union)")
    print(df.to_string(index=False))
    print()

Activity: 7 attributes (5 direct + 2 via union)
                            attribute           declaring_class range_kind
                    Activity-childIds                  Activity     direct
                      Activity-nextId                  Activity     direct
                  Activity-previousId                  Activity     direct
ScheduledActivityInstance-activityIds ScheduledActivityInstance     direct
               StudyDesign-activities               StudyDesign     direct
               Condition-appliesToIds                 Condition  via_union
                 Condition-contextIds                 Condition  via_union

BiomedicalConcept: 4 attributes (3 direct + 1 via union)
                          attribute           declaring_class range_kind
      Activity-biomedicalConceptIds                  Activity     direct
BiomedicalConceptCategory-memberIds BiomedicalConceptCategory     direct
    StudyVersion-biomedicalConcepts              StudyVersion     direct
  

Without `owl:unionOf` traversal, the `via_union` rows are invisible — a
consumer walking direct ranges only would miss that `Condition.appliesTo`
and `Condition.context` can hold an Activity, and that `Condition.appliesTo`
can hold a BiomedicalConcept. In RDF the polymorphism is a queryable graph
fact rather than something the consumer reconstructs from YAML `Type` lists.

## Step 3 — systematic reverse range index

The same query over every target class at once. One row per
(object property × target class × range kind), saved as
`polymorphic_reverse_index.csv` for downstream model navigation.

In [4]:
FULL_INDEX_SPARQL = """
PREFIX owl:  <http://www.w3.org/2002/07/owl#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?prop ?declaring ?target ?range_kind
WHERE {
  ?prop rdfs:domain ?declaring .
  {
    ?prop a owl:ObjectProperty ; rdfs:range ?target .
    FILTER(isIRI(?target))
    BIND("direct" AS ?range_kind)
  } UNION {
    ?prop a owl:ObjectProperty ; rdfs:range ?r .
    FILTER(isBlank(?r))
    ?r owl:unionOf ?list .
    ?list rdf:rest*/rdf:first ?target .
    BIND("via_union" AS ?range_kind)
  }
}
"""

rows = []
for r in g.query(FULL_INDEX_SPARQL):
    rows.append({
        "target_class":    short(r["target"]),
        "attribute":       short(r["prop"]),
        "declaring_class": short(r["declaring"]),
        "range_kind":      str(r["range_kind"]),
    })

reverse_index = (pd.DataFrame(rows)
                 .sort_values(["target_class", "range_kind", "attribute"])
                 .reset_index(drop=True))

n_direct = (reverse_index.range_kind == "direct").sum()
n_union  = (reverse_index.range_kind == "via_union").sum()
n_targets = reverse_index.target_class.nunique()

print(f"{len(reverse_index)} rows, {n_direct} direct + {n_union} via_union, {n_targets} distinct target classes")

reverse_index.to_csv("polymorphic_reverse_index.csv", index=False)
print("wrote polymorphic_reverse_index.csv")

reverse_index.head(15)

341 rows, 330 direct + 11 via_union, 80 distinct target classes
wrote polymorphic_reverse_index.csv


,target_class,attribute,declaring_class,range_kind
0,Abbreviation,StudyVersion-abbreviations,StudyVersion,direct
1,Activity,Activity-childIds,Activity,direct
2,Activity,Activity-nextId,Activity,direct
3,Activity,Activity-previousId,Activity,direct
4,Activity,ScheduledActivityInstance-activityIds,ScheduledActivityInstance,direct
5,Activity,StudyDesign-activities,StudyDesign,direct
6,Activity,Condition-appliesToIds,Condition,via_union
7,Activity,Condition-contextIds,Condition,via_union
8,Address,Organization-legalAddress,Organization,direct
9,AdministrableProduct,Administration-administrableProductId,Administration,direct


## Step 4 — classes reachable only via polymorphism

For each target class, count direct vs. via-union attributes. A class with
`direct == 0` and `via_union > 0` is reachable from object properties
*only* through polymorphism — a YAML walker that does not expand `Type`
union lists would not see any attribute carrying it.

In [5]:
by_target = (reverse_index
             .pivot_table(index="target_class", columns="range_kind",
                          values="attribute", aggfunc="count", fill_value=0)
             .reset_index())

for col in ("direct", "via_union"):
    if col not in by_target.columns:
        by_target[col] = 0

only_poly = (by_target[(by_target["direct"] == 0) & (by_target["via_union"] > 0)]
             [["target_class", "direct", "via_union"]]
             .reset_index(drop=True))

print(f"classes reachable only via polymorphism: {len(only_poly)}")
only_poly

classes reachable only via polymorphism: 1


range_kind,target_class,direct,via_union
0,ScheduledActivityInstance,0,1


## Discussion

USDM v4 declares four polymorphic associations on three classes
(`Condition` × 2, `ProductOrganizationRole` × 1, `StudyRole` × 1).
Rendering them as `owl:unionOf` ranges makes the resulting target
relationships queryable in a single SPARQL question rather than
reconstructed by walking YAML `Type` lists at the consumer.

**Scope of this notebook.** Object properties only. Datatype properties
(e.g. `xsd:string` ranges) and `owl:Restriction` cardinality blocks are
not in the reverse range index. Instance-level validation of USDM
documents against the model is out of scope and tracked under SHACL in
the v0.3 known-gaps document.

**Downstream use.** The reverse range index is the foundational lookup
for impact analysis ("if I change class X, which attributes are
affected?"), code generation ("which classes must be available before
this attribute can be populated?"), and model navigation tools that need
to traverse both direct and polymorphic relationships in one pass.